# Retail Case Study Project

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DecimalType, DateType

In [0]:
# current Spark version and catalog
print(f"Spark version: {spark.version}")
print(f"Current catalog: {spark.catalog.currentCatalog()}")

Spark version: 4.1.0
Current catalog: workspace


## Bronze Layer

In [0]:
VOLUME_PATH = "/Volumes/workspace/retail_fresher/retail_raw"
raw_customers = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/customers.csv")
raw_products = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/products.csv")
raw_sales_orders = spark.read.option("header", True).option("inferSchema", False).csv(f"{VOLUME_PATH}/sales_orders.csv")

In [0]:
bronze_customers = (
    raw_customers
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/customers.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_products = (
    raw_products
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/products.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_sales_orders = (
    raw_sales_orders
    .withColumn("source_file", F.lit(f"{VOLUME_PATH}/sales_orders.csv"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
)

bronze_customers.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_customers")
bronze_products.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_products")
bronze_sales_orders.write.format("delta").mode("overwrite").saveAsTable("workspace.retail_fresher.bronze_sales_orders")

## PySpark Transformations

### Step 1

In [0]:
customers = spark.read.table("workspace.retail_fresher.bronze_customers")
print("Bronze customer rows", customers.count())
customers.printSchema()

Bronze customer rows 500
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- signup_date: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- is_active: string (nullable = true)
 |-- loyalty_points: string (nullable = true)
 |-- updated_at: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
products = spark.read.table("workspace.retail_fresher.bronze_products")
print("Bronze product rows", products.count())
products.printSchema()

Bronze product rows 500
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- cost_price: string (nullable = true)
 |-- supplier_name: string (nullable = true)
 |-- stock_quantity: string (nullable = true)
 |-- launch_date: string (nullable = true)
 |-- product_rating: string (nullable = true)
 |-- active_flag: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



In [0]:
sales_orders = spark.read.table("workspace.retail_fresher.bronze_sales_orders")
print("Bronze sales order rows", sales_orders.count())
sales_orders.printSchema()

Bronze sales order rows 500
root
 |-- order_id: string (nullable = true)
 |-- order_timestamp: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- discount_pct: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- promised_delivery_date: string (nullable = true)
 |-- actual_delivery_date: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



### Steps 2 & 3

In [0]:
# customers
customers = (
    customers
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("customer_name", F.initcap(F.trim(F.col("customer_name"))))
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn(
        "city",
        F.when(F.col("city").isNull() | (F.col("city") == ""), "Unknown").otherwise(F.col("city"))
    )
    .withColumn("state", F.initcap(F.trim(F.col("state"))))
    .withColumn("region", F.initcap(F.trim(F.col("region"))))
    .withColumn("customer_segment", F.initcap(F.trim(F.col("customer_segment"))))
    .withColumn("is_active", F.upper(F.trim(F.col("is_active"))))
)

In [0]:
# products
products = (
    products
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))
    .withColumn("category", F.initcap(F.trim(F.col("category"))))
    .withColumn("subcategory", F.initcap(F.trim(F.col("subcategory"))))
    .withColumn("supplier_name", F.initcap(F.trim(F.col("supplier_name"))))
    .withColumn("active_flag", F.upper(F.trim(F.col("active_flag"))))
)

In [0]:
# sales orders
sales_orders = (
    sales_orders
    .withColumn("order_id", F.trim(F.col("order_id")))
    .withColumn("customer_id", F.trim(F.col("customer_id")))
    .withColumn("product_id", F.trim(F.col("product_id")))
    .withColumn("payment_method", F.upper(F.trim(F.col("payment_method"))))
    .withColumn("order_status", F.upper(F.trim(F.col("order_status"))))
    .withColumn("sales_channel", F.upper(F.trim(F.col("sales_channel"))))
    .withColumn("warehouse_id", F.trim(F.col("warehouse_id")))
)

### Step 4

In [0]:
# customers
customers = (
    customers
    .withColumn("signup_date", F.to_date(F.col("signup_date"), "yyyy-MM-dd"))
    .withColumn("date_of_birth", F.to_date(F.col("date_of_birth"), "yyyy-MM-dd"))
    .withColumn("updated_at", F.to_timestamp(F.col("updated_at"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("loyalty_points", F.col("loyalty_points").try_cast(IntegerType()))
)

In [0]:
# products
products = (
    products
    .withColumn("unit_price", F.col("unit_price").try_cast(DecimalType(10, 2)))
    .withColumn("cost_price", F.col("cost_price").try_cast(DecimalType(10, 2)))
    .withColumn("stock_quantity", F.col("stock_quantity").try_cast(IntegerType()))
    .withColumn("launch_date", F.to_date(F.col("launch_date"), "yyyy-MM-dd"))
    .withColumn("product_rating", F.col("product_rating").try_cast(DecimalType(3, 2)))
)

In [0]:
# sales_orders
sales_orders = (
    sales_orders
    .withColumn("order_timestamp", F.to_timestamp(F.col("order_timestamp"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("quantity", F.col("quantity").try_cast(IntegerType()))
    .withColumn("discount_pct", F.col("discount_pct").try_cast(DecimalType(5, 2)))
    .withColumn("promised_delivery_date", F.to_date(F.col("promised_delivery_date"), "yyyy-MM-dd"))
    .withColumn("actual_delivery_date", F.to_date(F.col("actual_delivery_date"), "yyyy-MM-dd"))
)

### Step 5 & 6

In [0]:
# steps 5 & 6 for customers
w = Window.partitionBy("customer_id").orderBy(F.desc_nulls_last("updated_at"))

deduped_customers = (
    customers
    .withColumn("row_num", F.row_number().over(w))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .filter(F.col("is_active") == F.lit("Y"))
)

### Step 7 & 8

In [0]:
products = (
    products
    .filter(
        (F.col("unit_price") > 0) &
        (F.col("cost_price") > 0)
    )
    .filter(F.col("active_flag") == F.lit("Y"))
)

### Step 9 & 10

In [0]:
sales_orders = (
    sales_orders
    .filter(F.col("quantity") > 0)
    .filter(~(F.col("order_status").isin(["CANCELLED", "PENDING"])))
)

### Step 11 & 12

In [0]:
# add delivery_days, late_delivery_flag, and order_month before joins
sales_orders = (
    sales_orders
    .withColumn("order_month", F.date_format(F.col("order_timestamp"), "yyyy-MM"))
    .withColumn(
        "late_delivery_flag",
        F.when(F.col("actual_delivery_date").isNull(), None)
        .when(F.col("actual_delivery_date") > F.col("promised_delivery_date"), F.lit("Y"))
        .otherwise(F.lit("N"))
    )
    .withColumn("delivery_days", F.datediff(F.col("actual_delivery_date"), F.col("order_timestamp")))
)

In [0]:
# join tables for remaining columns
full_table = (
    sales_orders.join(deduped_customers, "customer_id", "inner")
    .join(products, "product_id", "inner")
    .withColumn("gross_amount", F.col("quantity") * F.col("unit_price"))
    .withColumn("discount_amount", F.col("gross_amount") * F.coalesce(F.col("discount_pct"), F.lit(0)) / 100)
    .withColumn("net_amount", F.col("gross_amount") - F.col("discount_amount"))
    .withColumn("net_sales", F.when(F.col("order_status") == "COMPLETED", F.col("net_amount")).otherwise(F.lit(0)))
    .withColumn("profit_per_unit", (F.col("net_amount") / F.col("quantity")) - F.col("cost_price"))
    .drop("ingestion_timestamp", "source_file")
)

### Step 13

In [0]:
# save managed tables
customers.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_customers")
products.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_products")
sales_orders.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_sales_orders")
full_table.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.silver_full_table")

### Step 14

In [0]:
display(full_table.limit(5))

product_id,customer_id,order_id,order_timestamp,quantity,discount_pct,payment_method,order_status,promised_delivery_date,actual_delivery_date,sales_channel,warehouse_id,order_month,late_delivery_flag,delivery_days,customer_name,email,city,state,region,customer_segment,signup_date,date_of_birth,is_active,loyalty_points,updated_at,product_name,category,subcategory,unit_price,cost_price,supplier_name,stock_quantity,launch_date,product_rating,active_flag,gross_amount,discount_amount,net_amount,net_sales,profit_per_unit
P0030,C0018,O000001,2026-01-08T09:07:00.000Z,2,5.00,CARD,COMPLETED,2026-01-12,2026-01-11,MOBILE,WH02,2026-01,N,3,Vikram Menon,vikram.menon18@example.com,Delhi,Delhi,North,Consumer,2023-07-18,1986-07-19,Y,666,2026-01-03T14:00:00.000Z,Football Model 030,Sports,Sports Sub-3,669.00,434.85,Nova Supply,140,2022-09-28,3.50,Y,1338.00,66.90000000,1271.1000000,1271.1000000,200.700000
P0059,C0035,O000002,2026-01-15T10:14:00.000Z,3,10.00,CASH,COMPLETED,2026-01-19,2026-01-19,STORE,WH03,2026-01,N,4,Sneha Davis,sneha.davis35@example.com,Pune,Maharashtra,West,Home Office,2024-01-21,2003-12-09,Y,1295,2026-01-05T17:00:00.000Z,Table Model 059,Home,Home Sub-4,1170.70,866.32,Metro Wholesale,17,2023-06-16,3.80,Y,3512.10,351.21000000,3160.8900000,3160.8900000,187.310000
P0088,C0052,O000003,2026-01-22T11:21:00.000Z,4,15.00,NET_BANKING,COMPLETED,2026-01-26,2026-01-27,WEB,WH04,2026-01,Y,5,Naveen Patel,naveen.patel52@example.com,Mumbai,Maharashtra,West,Corporate,2024-07-26,1983-05-26,Y,1924,2026-01-07T20:00:00.000Z,Tea Model 088,Grocery,Grocery Sub-1,1672.40,1053.61,Vertex Goods,144,2024-03-03,4.10,Y,6689.60,1003.44000000,5686.1600000,5686.1600000,367.930000
P0117,C0069,O000004,2026-01-29T12:28:00.000Z,5,20.00,UPI,COMPLETED,2026-02-02,2026-02-04,MOBILE,WH05,2026-01,Y,6,Asha Reddy,asha.reddy69@example.com,Kochi,Kerala,South,Consumer,2025-01-29,2000-10-16,Y,2553,2026-01-09T23:00:00.000Z,Shoes Model 117,Fashion,Fashion Sub-2,2174.10,1565.35,Global Mart,21,2024-11-19,4.40,Y,10870.50,2174.10000000,8696.4000000,8696.4000000,173.930000
P0146,C0086,O000005,2026-02-05T13:35:00.000Z,1,0.00,CARD,COMPLETED,2026-02-09,2026-02-12,STORE,WH01,2026-02,Y,7,Liam Brown,liam.brown86@example.com,Hyderabad,Telangana,South,Home Office,2023-02-16,1980-03-06,Y,3182,2026-01-12T02:00:00.000Z,Headset Model 146,Electronics,Electronics Sub-3,2675.80,1632.24,Prime Traders,148,2022-04-25,4.70,Y,2675.80,0E-8,2675.8000000,2675.8000000,1043.560000


In [0]:
# monthly category sales
monthly_cat_sales = (
    full_table
    .select("order_month", "category", "net_sales")
    .groupBy("order_month", "category")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy("order_month", F.col("total_sales").desc())
)
monthly_cat_sales.show(truncate = False)

+-----------+-----------+-----------+
|order_month|category   |total_sales|
+-----------+-----------+-----------+
|2026-01    |Fashion    |239436.00  |
|2026-01    |Grocery    |159306.66  |
|2026-01    |Home       |139416.12  |
|2026-01    |Sports     |95083.60   |
|2026-01    |Electronics|50978.00   |
|2026-02    |Grocery    |237036.28  |
|2026-02    |Fashion    |188388.40  |
|2026-02    |Home       |125620.74  |
|2026-02    |Sports     |93691.85   |
|2026-02    |Electronics|44317.50   |
|2026-03    |Fashion    |240081.60  |
|2026-03    |Grocery    |222273.84  |
|2026-03    |Home       |161852.85  |
|2026-03    |Sports     |108117.30  |
|2026-03    |Electronics|29364.40   |
|2026-04    |Fashion    |182385.10  |
|2026-04    |Grocery    |149993.04  |
|2026-04    |Home       |112417.20  |
|2026-04    |Sports     |108333.55  |
|2026-04    |Electronics|31786.40   |
+-----------+-----------+-----------+
only showing top 20 rows


In [0]:
# city sales
city_sales = (
    full_table
    .select("city", "net_sales")
    .groupBy("city")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("total_sales").desc())
)

city_sales.show(truncate = False)

+-----------+-----------+
|city       |total_sales|
+-----------+-----------+
|Kolkata    |646707.20  |
|Kochi      |604234.70  |
|Bhubaneswar|547022.44  |
|Mumbai     |535789.20  |
|Pune       |474324.24  |
|Chennai    |380696.64  |
|Bengaluru  |305162.15  |
|Delhi      |303587.40  |
|Hyderabad  |159992.80  |
|Jaipur     |72211.40   |
|Unknown    |48056.47   |
+-----------+-----------+



In [0]:
# customer value
customer_value = (
    full_table
    .select("customer_id", "customer_name", "net_sales")
    .groupBy("customer_id", "customer_name")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("total_sales").desc())
)

customer_value.show(truncate = False)

+-----------+-------------+-----------+
|customer_id|customer_name|total_sales|
+-----------+-------------+-----------+
|C0154      |Vikram Rao   |51992.80   |
|C0137      |Meera Ram    |40782.32   |
|C0379      |Sneha Iyer   |34992.40   |
|C0074      |Vikram Rao   |34646.40   |
|C0274      |Vikram Menon |33954.40   |
|C0169      |Meera Ram    |32916.40   |
|C0354      |Vikram Menon |32570.40   |
|C0049      |Meera Singh  |32224.40   |
|C0257      |Meera Singh  |31947.60   |
|C0249      |Meera Ram    |31532.40   |
|C0144      |Aarav Kumar  |30494.40   |
|C0329      |Meera Ram    |30148.40   |
|C0344      |Aarav Verma  |29802.40   |
|C0082      |Vikram Menon |29802.36   |
|C0267      |Sneha Iyer   |29508.26   |
|C0282      |Vikram Rao   |29214.16   |
|C0224      |Aarav Kumar  |29110.40   |
|C0239      |Sophia Shah  |28764.40   |
|C0162      |Vikram Menon |28625.96   |
|C0362      |Vikram Rao   |28037.76   |
+-----------+-------------+-----------+
only showing top 20 rows


In [0]:
# top products by category
top_products = (
    full_table
    .select("product_id", "product_name", "category", "net_sales")
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.round(F.sum("net_sales"), 2).alias("total_sales")
    )
    .orderBy(F.col("category"), F.col("total_sales").desc())
)

top_products.show(truncate = False)

+----------+------------------+-----------+-----------+
|product_id|product_name      |category   |total_sales|
+----------+------------------+-----------+-----------+
|P0496     |Headset Model 496 |Electronics|8730.80    |
|P0491     |Mouse Model 491   |Electronics|8644.30    |
|P0486     |Keyboard Model 486|Electronics|8557.80    |
|P0476     |Laptop Model 476  |Electronics|8384.80    |
|P0471     |Headset Model 471 |Electronics|8298.30    |
|P0456     |Monitor Model 456 |Electronics|8038.80    |
|P0451     |Laptop Model 451  |Electronics|7952.30    |
|P0446     |Headset Model 446 |Electronics|7865.80    |
|P0436     |Keyboard Model 436|Electronics|7692.80    |
|P0416     |Mouse Model 416   |Electronics|7346.80    |
|P0411     |Keyboard Model 411|Electronics|7260.30    |
|P0406     |Monitor Model 406 |Electronics|7173.80    |
|P0376     |Laptop Model 376  |Electronics|6654.80    |
|P0371     |Headset Model 371 |Electronics|6568.30    |
|P0366     |Mouse Model 366   |Electronics|6481.

In [0]:
# save gold tables
monthly_cat_sales.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_monthly_cat_sales")
city_sales.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_city_sales")
customer_value.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_customer_value")
top_products.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("workspace.retail_fresher.gold_top_products")

## Part C - Aggregation & Window Functions